# Embedding

Logan Wong

law3082

I followed the README in twitter-incremental-clustering.

preprocess.py cleaned the raw data into a readable .tsv

THIS notebook loads this .tsv file
Runs it through SBERT model
Produces a matrix of shape (N, 768)

In [5]:
import os
import pandas as pd
from sentence_transformers import SentenceTransformer
import numpy as np

In [2]:
# Redirect HuggingFace cache to avoid Disk Quota Exceeded errors
os.environ['HF_HOME'] = '/tmp/law3082_hf'

In [6]:
# Load the data generated by preprocess.py
df = pd.read_csv('./twitter-incremental-clustering/event2012.tsv', sep='\t')

# Display the first few rows to confirm: id, text, created_at, label
print(f"Loaded {len(df)} tweets.")
df.head()

Loaded 68841 tweets.


,id,text,created_at,label
0,256292946331181056,Nobel prize in literature to be announced http...,2012-10-11 07:19:34,0
1,256333064467279872,“@marvicleonen: Is it true that UP won UAAP ba...,2012-10-11 09:58:59,0
2,256334302034399232,"Congrats, Ateneo! Last na yan ha. Season 76 wi...",2012-10-11 10:03:54,0
3,256335853738160128,"""@SMARTPromos: SMART never wants you to be lef...",2012-10-11 10:10:04,0
4,256346272506712064,CCTV invite hints at Nobel literature prize fo...,2012-10-11 10:51:28,0


In [7]:
# Init the model
# download to /tmp/law3082_hf if not already there
model_name = "sentence-transformers/all-mpnet-base-v2"
model = SentenceTransformer(model_name)

# Convert 'text' column to embeddings
print("Encoding tweets... this may take a few minutes.")
embeddings = model.encode(df['text'].tolist(), 
                          batch_size=32, 
                          show_progress_bar=True, 
                          convert_to_numpy=True)

print(f"Embedding shape: {embeddings.shape}")

Encoding tweets... this may take a few minutes.


Batches:   0%|          | 0/2152 [00:00<?, ?it/s]

Embedding shape: (68841, 768)


In [8]:
# Save embeddings & corresponding IDs for use in DPAC
np.save('data/event2012_embeddings.npy', embeddings)
df[['id', 'label']].to_csv('data/event2012_metadata.csv', index=False)

print("Embeddings saved to event2012_embeddings.npy")

Embeddings saved to event2012_embeddings.npy
